# 11 · GPT Modeli ve Sonraki Token

**Belgedeki bölüm:** Transformer Mimarisi › 4. Basamak: Linear + Softmax, GPT Gibi Modeller

- UstaModel = Embedding (+RoPE) → N tane decoder bloğu → Linear (lm_head).
- Linear: her token'ın vektörünü sözlük boyutunda skorlara (logits) çevirir; softmax ile olasılık olur.
- Son pozisyondaki en olası token seçilir, sona eklenir ve döngü tekrarlar (next-token prediction).
- Model henüz eğitilmediği için tahminler rastgele; eğitim 12'de.

In [ ]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)
tokens

In [ ]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, num_heads=2, context_length=32, num_layers=3)

out = u_model(tokens)
out.shape

In [ ]:
u_model

In [ ]:
u_model.eval()  # tahmin yaparken dropout kapalı

In [ ]:
import torch

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

In [ ]:
u_tokenizer.decode([max_index.item()])  # tahmin edilen sonraki token

### Metin üretimi (generate)

In [ ]:
generated = u_model.generate(tokens, max_new_tokens=5)
u_tokenizer.decode(generated)

## İsteğe bağlı: Aynı adımlar gerçek bir modelde (Qwen3 0.6B)
Model ~1.2 GB indirir.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

In [ ]:
q_tokens = q_tokenizer.encode(prompt)
q_tokens

In [ ]:
q_model.generate(torch.tensor([q_tokens]))

In [ ]:
# input = [1782, 6722, 315, 28192]
# output = [38297, 315, 279, 5302]
# expected = [6722, 315, 28192, 5302]

In [ ]:
q_out = q_model(torch.tensor([q_tokens]))
q_out

In [ ]:
q_out.logits.shape

In [ ]:
q_out.logits[0, 0, :].shape

In [ ]:
probs = torch.softmax(q_out.logits[0, 2, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

In [ ]:
q_tokenizer.decode([max_index])